In [1]:
from importlib.metadata import version

pkgs = [
    "numpy",       # PyTorch & TensorFlow dependency
    "matplotlib",  # Plotting library
    "tiktoken",    # Tokenizer
    "torch",       # Deep learning library
    "tqdm",        # Progress bar
    "tensorflow",  # For OpenAI's pretrained weights
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

numpy version: 1.26.4
matplotlib version: 3.10.6
tiktoken version: 0.12.0
torch version: 2.2.2
tqdm version: 4.66.5
tensorflow version: 2.16.2


In [2]:
import json
import os
import requests

In [3]:
def download_and_load_file(file_path, url):
	if not os.path.exists(file_path):
		response = requests.get(url, timeout=30)
		response.raise_for_status()
		text_data = response.text
		with open(file_path, "w", encoding="utf-8") as file:
			file.write(text_data)

	with open(file_path, "r", encoding="utf-8") as file:
		data = json.load(file)

	return data

In [4]:
file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


In [5]:
print("Example entry:\n", data[50])
print("Another example entry:\n", data[999])

Example entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}
Another example entry:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


In [6]:
def format_input(entry):
	instruction_text = (
		f"Below is an instruction that describes a task. "
		f"Write a response that appropriately completes the request."
		f"\n\n### Instruction:\n{entry["instruction"]}"
	)

	input_text = f"\n\n### Input:\n{entry["input"]}" if entry["input"] else ""

	return instruction_text + input_text

In [7]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]["output"]}"

print(model_input + desired_response)

print("\n\n###################")
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


###################
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


In [8]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[: train_portion]
test_data = data[train_portion: train_portion+test_portion]
val_data = data[train_portion+test_portion: ]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


In [6]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
	def __init__(self,data,tokenizer):
		super().__init__()

		self.data = data
		self.encoded_texts = []
		for entry in data:
			instruction_plus_input = format_input(entry)
			response_text = f"\n\n### Response:\n{entry["output"]}"
			full_text = instruction_plus_input + response_text
			self.encoded_texts.append(
				tokenizer.encode(full_text)
			)
	
	def __getitem__(self, index):
		return self.encoded_texts[index]

	def __len__(self):
		return len(self.data)

In [2]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [4]:
def custom_collate_draft_1(batch,pad_token_id = 50256,device="cpu"):
	batch_max_length = max(len(item) + 1 for item in batch)
	inputs_lst = []

	for item in batch:
		new_item = item.copy()
		new_item += [pad_token_id]

		padded = (new_item + [pad_token_id]*(batch_max_length-len(new_item)))
		inputs = torch.tensor(padded[:-1])
		inputs_lst.append(inputs)
	
	inputs_tensor = torch.stack(inputs_lst).to(device)
	return inputs_tensor

In [7]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
